In [1]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
# 🔒 Disable W&B
os.environ["WANDB_DISABLED"] = "true"

In [4]:
# 📥 Load dataset
df = pd.read_csv('./product_review.csv')  # Make sure this file is in the same directory

In [5]:
print(df.shape)

(205052, 2)


In [6]:
# 🧹 Clean and encode
df = df[['review', 'label']].dropna()
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

In [7]:
# Randomly sample 5000 rows
df = df.sample(n=1500, random_state=42).reset_index(drop=True)


In [8]:
# 🔀 Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

In [9]:
print(df.shape)

(1500, 2)


In [10]:
# 🔤 Tokenization
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

In [11]:
# 🧱 Dataset wrapper
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)

In [ ]:
# 🧠 Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# ⚙️ Training config
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch"

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [14]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 📊 Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:

# 🧰 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics 
)

# 🚀 Train
trainer.train()



C:\Users\91739\AppData\Local\Temp\ipykernel_10204\1721501943.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.239500,0.322337,0.920000,0.897687,0.920000,0.905545
2,0.062700,0.346802,0.920000,0.897687,0.920000,0.905545
3,0.143800,0.313584,0.920000,0.907194,0.920000,0.912233


c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Model saved to './bert-finetuned-final'


In [19]:
# 💾 Save final model
trainer.save_model('./bert-finetuned-final')
print("✅ Model saved to './bert-finetuned-final'")
tokenizer.save_pretrained("./bert-finetuned-final")

# 🔍 Inference
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_class])[0]


✅ Model saved to './bert-finetuned-final'


In [20]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained("./bert-finetuned-final")


('./bert-finetuned-final\\tokenizer_config.json',
 './bert-finetuned-final\\special_tokens_map.json',
 './bert-finetuned-final\\vocab.txt',
 './bert-finetuned-final\\added_tokens.json',
 './bert-finetuned-final\\tokenizer.json')

In [21]:


import joblib
joblib.dump(label_encoder, "label_encoder.pkl")


['label_encoder.pkl']

In [17]:
# 📈 Evaluate on validation set
results = trainer.evaluate()
print(results)


c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.31358352303504944, 'eval_accuracy': 0.92, 'eval_precision': 0.9071942257217848, 'eval_recall': 0.92, 'eval_f1': 0.9122325718932507, 'eval_runtime': 10.579, 'eval_samples_per_second': 28.358, 'eval_steps_per_second': 3.592, 'epoch': 3.0}


In [ ]:
from ipywidgets import Text, Button, VBox, Output
from IPython.display import display

out = Output()

text_box = Text(
    value='',
    placeholder='Type your product review here...',
    description='Review:',
    disabled=False
)

button = Button(description="Predict Sentiment")

def on_button_clicked(b):
    review = text_box.value
    sentiment = predict_sentiment(review)
    with out:
        out.clear_output()
        print(f"Predicted sentiment: {sentiment}")

button.on_click(on_button_clicked)

display(VBox([text_box, button, out]))
